In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import time

# Create Spark session with performance monitoring enabled
spark = SparkSession.builder \
    .appName("Day4_Performance_Optimization") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .config("spark.sql.adaptive.localShuffleReader.enabled", "true") \
    .config("spark.sql.files.maxPartitionBytes", "128MB") \
    .config("spark.ui.port", "4040") \
    .getOrCreate()


spark.sparkContext.setLogLevel("WARN")
print(f"✅ Spark session created: {spark.version}")

26/03/10 19:57:47 WARN Utils: Your hostname, odinsbeard-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/03/10 19:57:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/10 19:58:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/10 19:58:08 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/10 19:58:08 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/03/10 19:58:08 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


✅ Spark session created: 3.5.0


In [2]:
# Load data
df_sales = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv("file:///home/odinsbeard/Data_engineering_Journey/week6_spark/data/input/sales_*.csv")

df_products = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv("file:///home/odinsbeard/Data_engineering_Journey/week6_spark/data/input/products_*.csv")

df_users = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv("file:///home/odinsbeard/Data_engineering_Journey/week6_spark/data/input/users_*.csv")


# Cache the data for repeated use
df_sales.cache()
df_products.cache()
df_users.cache()


# Force cache (action).......
row_count = df_sales.count()
print(f"✅ Data loaded and cached: {row_count:,} sales rows")
print(f"   • Products: {df_products.count():,}")
print(f"   • Users: {df_users.count():,}")

26/03/10 20:02:03 WARN MemoryStore: Not enough space to cache rdd_33_3 in memory! (computed 56.7 MiB so far)
26/03/10 20:02:03 WARN BlockManager: Persisting block rdd_33_3 to disk instead.
26/03/10 20:02:03 WARN MemoryStore: Not enough space to cache rdd_33_1 in memory! (computed 56.7 MiB so far)
26/03/10 20:02:03 WARN BlockManager: Persisting block rdd_33_1 to disk instead.
26/03/10 20:02:15 WARN MemoryStore: Not enough space to cache rdd_33_10 in memory! (computed 5.4 MiB so far)
26/03/10 20:02:15 WARN BlockManager: Persisting block rdd_33_10 to disk instead.
26/03/10 20:02:17 WARN MemoryStore: Not enough space to cache rdd_33_8 in memory! (computed 10.6 MiB so far)
26/03/10 20:02:17 WARN BlockManager: Persisting block rdd_33_8 to disk instead.
26/03/10 20:02:25 WARN MemoryStore: Not enough space to cache rdd_33_6 in memory! (computed 20.8 MiB so far)
26/03/10 20:02:25 WARN BlockManager: Persisting block rdd_33_6 to disk instead.
26/03/10 20:02:26 WARN MemoryStore: Not enough space t

✅ Data loaded and cached: 20,000,000 sales rows


   • Products: 400,000


[Stage 14:================================================>         (5 + 1) / 6]

   • Users: 4,000,000


In [6]:
# ============================================================================
# PROPER PARTITION OPTIMIZATION
# ============================================================================

# Get row count as Python integer
total_rows = df_sales.count()
print(f"📊 Total rows: {total_rows:,}")

# Get current partitions....
current_partitions = df_sales.rdd.getNumPartitions()
print(f"📊 Current partitions: {current_partitions}")
print(f"   • Current rows/partition: {total_rows // current_partitions:,}")

# OPTIMAL PARTITION CALCULATION
# Aim for 200-500MB per partition or 500k-1M rows
# Formula: target = total_rows / desired_rows_per_partition

desired_rows_per_partition = 500000  # 500k rows per partition
calculated_partitions = total_rows // desired_rows_per_partition

# Minimum partitions (should be at least number of cores)
min_partitions = 8  # At least 8 partitions for parallelism
max_partitions = 200  # Upper limit to avoid too many small partitions

# Calculate optimal partitions
if calculated_partitions < min_partitions:
    optimal_partitions = min_partitions
    reason = "Minimum partitions (for parallelism)"
elif calculated_partitions > max_partitions:
    optimal_partitions = max_partitions
    reason = "Maximum partitions (avoid overhead)"
else:
    optimal_partitions = calculated_partitions
    reason = f"{desired_rows_per_partition:,} rows per partition"

print(f"\n🎯 Optimal partitions: {optimal_partitions}")
print(f"   • Reason: {reason}")
print(f"   • Expected rows/partition: {total_rows // optimal_partitions:,}")


# Perform repartition
print(f"\n🔄 Repartitioning to {optimal_partitions} partitions...")
df_optimized = df_sales.repartition(optimal_partitions)


# Verify new partitions
new_partitions = df_optimized.rdd.getNumPartitions()
print(f"✅ New partitions: {new_partitions}")
print(f"✅ New rows/partition: {total_rows // new_partitions:,}")


# Show partition distribution
print("\n📈 New partition distribution:")
df_optimized.withColumn("partition_id", spark_partition_id()) \
    .groupBy("partition_id") \
    .count() \
    .orderBy("partition_id") \
    .show(new_partitions[:10], truncate=False)

26/03/10 20:38:02 WARN MemoryStore: Not enough space to cache rdd_33_0 in memory! (computed 36.2 MiB so far)
26/03/10 20:38:03 WARN MemoryStore: Not enough space to cache rdd_33_3 in memory! (computed 36.2 MiB so far)
26/03/10 20:38:03 WARN MemoryStore: Not enough space to cache rdd_33_4 in memory! (computed 36.2 MiB so far)
26/03/10 20:38:05 WARN MemoryStore: Not enough space to cache rdd_33_2 in memory! (computed 56.7 MiB so far)
                                                                                

📊 Total rows: 20,000,000
📊 Current partitions: 12
   • Current rows/partition: 1,666,666

🎯 Optimal partitions: 40
   • Reason: 500,000 rows per partition
   • Expected rows/partition: 500,000

🔄 Repartitioning to 40 partitions...


26/03/10 20:38:08 WARN MemoryStore: Not enough space to cache rdd_33_3 in memory! (computed 20.8 MiB so far)
26/03/10 20:38:08 WARN MemoryStore: Not enough space to cache rdd_33_0 in memory! (computed 20.8 MiB so far)
26/03/10 20:38:09 WARN MemoryStore: Not enough space to cache rdd_33_2 in memory! (computed 20.8 MiB so far)
26/03/10 20:38:09 WARN MemoryStore: Not enough space to cache rdd_33_4 in memory! (computed 20.8 MiB so far)
26/03/10 20:38:59 WARN MemoryStore: Not enough space to cache rdd_33_9 in memory! (computed 5.4 MiB so far)
26/03/10 20:38:59 WARN MemoryStore: Not enough space to cache rdd_33_8 in memory! (computed 20.8 MiB so far)
26/03/10 20:38:59 WARN MemoryStore: Failed to reserve initial memory threshold of 1024.0 KiB for computing block rdd_33_10 in memory.
26/03/10 20:38:59 WARN MemoryStore: Not enough space to cache rdd_33_10 in memory! (computed 384.0 B so far)
26/03/10 20:39:00 WARN MemoryStore: Failed to reserve initial memory threshold of 1024.0 KiB for computi

✅ New partitions: 40
✅ New rows/partition: 500,000

📈 New partition distribution:


TypeError: 'int' object is not subscriptable

In [7]:
# ============================================================================
# PROPER PARTITION OPTIMIZATION - COMPLETE WORKING VERSION
# ============================================================================

# Get row count as Python integer
total_rows = df_sales.count()
print(f"📊 Total rows: {total_rows:,}")

# Get current partitions
current_partitions = df_sales.rdd.getNumPartitions()
print(f"📊 Current partitions: {current_partitions}")
print(f"   • Current rows/partition: {total_rows // current_partitions:,}")

# OPTIMAL PARTITION CALCULATION
# Aim for 500k rows per partition (good balance)
desired_rows_per_partition = 500000
calculated_partitions = total_rows // desired_rows_per_partition

# Set boundaries
min_partitions = 8   # At least 8 for parallelism
max_partitions = 200 # Upper limit to avoid overhead

# Calculate optimal partitions using if-else (avoids Spark max() issue)
if calculated_partitions < min_partitions:
    optimal_partitions = min_partitions
    reason = f"Minimum partitions ({min_partitions}) - for parallelism"
elif calculated_partitions > max_partitions:
    optimal_partitions = max_partitions
    reason = f"Maximum partitions ({max_partitions}) - avoid overhead"
else:
    optimal_partitions = calculated_partitions
    reason = f"{desired_rows_per_partition:,} rows per partition"

print(f"\n🎯 Optimal partitions: {optimal_partitions}")
print(f"   • Reason: {reason}")
print(f"   • Expected rows/partition: {total_rows // optimal_partitions:,}")

# Perform repartition
print(f"\n🔄 Repartitioning to {optimal_partitions} partitions...")
df_optimized = df_sales.repartition(optimal_partitions)

# Verify new partitions
new_partitions = df_optimized.rdd.getNumPartitions()
print(f"✅ New partitions: {new_partitions}")
print(f"✅ New rows/partition: {total_rows // new_partitions:,}")

# Show first 10 partitions distribution
print("\n📈 New partition distribution (first 10):")
df_optimized.withColumn("partition_id", spark_partition_id()) \
    .groupBy("partition_id") \
    .count() \
    .orderBy("partition_id") \
    .show(10, truncate=False)

# Optional: Show partition summary statistics
print("\n📊 Partition Summary Statistics:")
df_optimized.withColumn("partition_id", spark_partition_id()) \
    .groupBy("partition_id") \
    .count() \
    .select(
        count("partition_id").alias("total_partitions"),
        min("count").alias("min_rows_per_partition"),
        max("count").alias("max_rows_per_partition"),
        round(avg("count"), 0).alias("avg_rows_per_partition")
    ).show()

26/03/10 20:49:14 WARN MemoryStore: Not enough space to cache rdd_33_5 in memory! (computed 36.2 MiB so far)
26/03/10 20:49:15 WARN MemoryStore: Not enough space to cache rdd_33_8 in memory! (computed 36.2 MiB so far)
26/03/10 20:49:16 WARN MemoryStore: Not enough space to cache rdd_33_3 in memory! (computed 56.7 MiB so far)
26/03/10 20:49:17 WARN MemoryStore: Failed to reserve initial memory threshold of 1024.0 KiB for computing block rdd_33_11 in memory.
26/03/10 20:49:17 WARN MemoryStore: Not enough space to cache rdd_33_11 in memory! (computed 384.0 B so far)
26/03/10 20:49:18 WARN MemoryStore: Not enough space to cache rdd_33_10 in memory! (computed 20.8 MiB so far)
                                                                                

📊 Total rows: 20,000,000
📊 Current partitions: 12
   • Current rows/partition: 1,666,666

🎯 Optimal partitions: 40
   • Reason: 500,000 rows per partition
   • Expected rows/partition: 500,000

🔄 Repartitioning to 40 partitions...


26/03/10 20:49:20 WARN MemoryStore: Not enough space to cache rdd_33_5 in memory! (computed 5.4 MiB so far)
26/03/10 20:49:20 WARN MemoryStore: Not enough space to cache rdd_33_3 in memory! (computed 10.6 MiB so far)
26/03/10 20:50:12 WARN MemoryStore: Not enough space to cache rdd_33_7 in memory! (computed 56.7 MiB so far)
26/03/10 20:50:12 WARN MemoryStore: Not enough space to cache rdd_33_11 in memory! (computed 20.8 MiB so far)
26/03/10 20:50:12 WARN MemoryStore: Not enough space to cache rdd_33_10 in memory! (computed 35.6 MiB so far)
[Stage 44:===================================================>    (11 + 1) / 12]

✅ New partitions: 40
✅ New rows/partition: 500,000

📈 New partition distribution (first 10):


26/03/10 20:50:53 WARN MemoryStore: Not enough space to cache rdd_33_0 in memory! (computed 36.2 MiB so far)
26/03/10 20:50:53 WARN MemoryStore: Not enough space to cache rdd_33_2 in memory! (computed 36.2 MiB so far)
26/03/10 20:50:55 WARN MemoryStore: Not enough space to cache rdd_33_5 in memory! (computed 56.7 MiB so far)
26/03/10 20:50:55 WARN MemoryStore: Not enough space to cache rdd_33_1 in memory! (computed 56.7 MiB so far)
26/03/10 20:51:03 WARN MemoryStore: Not enough space to cache rdd_33_9 in memory! (computed 36.2 MiB so far)
                                                                                

+------------+------+
|partition_id|count |
+------------+------+
|0           |499999|
|1           |500000|
|2           |500000|
|3           |500000|
|4           |500000|
|5           |500000|
|6           |500000|
|7           |500000|
|8           |500000|
|9           |500000|
+------------+------+
only showing top 10 rows


📊 Partition Summary Statistics:


26/03/10 20:51:11 WARN MemoryStore: Not enough space to cache rdd_33_1 in memory! (computed 36.2 MiB so far)
26/03/10 20:51:11 WARN MemoryStore: Not enough space to cache rdd_33_0 in memory! (computed 36.2 MiB so far)
26/03/10 20:51:13 WARN MemoryStore: Not enough space to cache rdd_33_5 in memory! (computed 56.7 MiB so far)
26/03/10 20:51:13 WARN MemoryStore: Not enough space to cache rdd_33_4 in memory! (computed 56.7 MiB so far)
26/03/10 20:51:21 WARN MemoryStore: Not enough space to cache rdd_33_8 in memory! (computed 36.2 MiB so far)
[Stage 53:==================================================>     (36 + 4) / 40]

+----------------+----------------------+----------------------+----------------------+
|total_partitions|min_rows_per_partition|max_rows_per_partition|avg_rows_per_partition|
+----------------+----------------------+----------------------+----------------------+
|              40|                499998|                500002|              500000.0|
+----------------+----------------------+----------------------+----------------------+



In [8]:
import time

# Original query (current partitions)
print("🔍 Query with current partitions:")
start = time.time()
df_sales.groupBy("country").count().show()
print(f"⏱️  Time: {time.time()-start:.2f} seconds")

# Repartition (full shuffle - expensive but balanced)
df_repartitioned = df_sales.repartition(16)
print(f"\n🔄 Repartitioned to 16 partitions: {df_repartitioned.rdd.getNumPartitions()}")

start = time.time()
df_repartitioned.groupBy("country").count().show()
print(f"⏱️  Time after repartition: {time.time()-start:.2f} seconds")

# Coalesce (no shuffle - only reduces partitions)
df_coalesced = df_sales.coalesce(4)
print(f"\n🔽 Coalesced to 4 partitions: {df_coalesced.rdd.getNumPartitions()}")

start = time.time()
df_coalesced.groupBy("country").count().show()
print(f"⏱️  Time after coalesce: {time.time()-start:.2f} seconds")

🔍 Query with current partitions:


26/03/10 22:34:55 WARN MemoryStore: Not enough space to cache rdd_33_3 in memory! (computed 36.2 MiB so far)
26/03/10 22:34:55 WARN MemoryStore: Not enough space to cache rdd_33_2 in memory! (computed 36.2 MiB so far)
26/03/10 22:34:55 WARN MemoryStore: Not enough space to cache rdd_33_4 in memory! (computed 36.2 MiB so far)
26/03/10 22:35:02 WARN MemoryStore: Not enough space to cache rdd_33_6 in memory! (computed 36.2 MiB so far)
26/03/10 22:35:02 WARN MemoryStore: Not enough space to cache rdd_33_8 in memory! (computed 36.2 MiB so far)
                                                                                

+---------+-------+
|  country|  count|
+---------+-------+
|  Germany|2500462|
|   France|2500495|
|      USA|2498441|
|       UK|2501002|
|   Canada|2501886|
|   Brazil|2501094|
|    Japan|2498267|
|Australia|2498353|
+---------+-------+

⏱️  Time: 14.97 seconds


26/03/10 22:35:05 WARN MemoryStore: Not enough space to cache rdd_33_4 in memory! (computed 10.6 MiB so far)
26/03/10 22:35:06 WARN MemoryStore: Not enough space to cache rdd_33_3 in memory! (computed 20.8 MiB so far)
26/03/10 22:35:06 WARN MemoryStore: Not enough space to cache rdd_33_2 in memory! (computed 20.8 MiB so far)
26/03/10 22:35:59 WARN MemoryStore: Not enough space to cache rdd_33_8 in memory! (computed 36.2 MiB so far)
26/03/10 22:35:59 WARN MemoryStore: Not enough space to cache rdd_33_7 in memory! (computed 36.2 MiB so far)
26/03/10 22:36:00 WARN MemoryStore: Failed to reserve initial memory threshold of 1024.0 KiB for computing block rdd_33_9 in memory.
26/03/10 22:36:00 WARN MemoryStore: Not enough space to cache rdd_33_9 in memory! (computed 384.0 B so far)
26/03/10 22:36:01 WARN MemoryStore: Failed to reserve initial memory threshold of 1024.0 KiB for computing block rdd_33_10 in memory.
26/03/10 22:36:01 WARN MemoryStore: Not enough space to cache rdd_33_10 in memor


🔄 Repartitioned to 16 partitions: 16


26/03/10 22:36:44 WARN MemoryStore: Not enough space to cache rdd_33_2 in memory! (computed 36.2 MiB so far)
26/03/10 22:36:44 WARN MemoryStore: Not enough space to cache rdd_33_0 in memory! (computed 36.2 MiB so far)
26/03/10 22:36:49 WARN MemoryStore: Not enough space to cache rdd_33_6 in memory! (computed 56.7 MiB so far)
26/03/10 22:36:55 WARN MemoryStore: Not enough space to cache rdd_33_7 in memory! (computed 56.7 MiB so far)
                                                                                

+---------+-------+
|  country|  count|
+---------+-------+
|  Germany|2500462|
|   France|2500495|
|      USA|2498441|
|       UK|2501002|
|   Canada|2501886|
|   Brazil|2501094|
|    Japan|2498267|
|Australia|2498353|
+---------+-------+

⏱️  Time after repartition: 20.81 seconds

🔽 Coalesced to 4 partitions: 4


26/03/10 22:37:05 WARN MemoryStore: Not enough space to cache rdd_33_2 in memory! (computed 56.7 MiB so far)
26/03/10 22:37:08 WARN MemoryStore: Not enough space to cache rdd_33_6 in memory! (computed 20.8 MiB so far)
26/03/10 22:37:08 WARN MemoryStore: Not enough space to cache rdd_33_1 in memory! (computed 36.2 MiB so far)
26/03/10 22:37:08 WARN MemoryStore: Not enough space to cache rdd_33_3 in memory! (computed 5.4 MiB so far)
[Stage 71:===========================================>              (3 + 1) / 4]

+---------+-------+
|  country|  count|
+---------+-------+
|  Germany|2500462|
|   France|2500495|
|      USA|2498441|
|       UK|2501002|
|   Canada|2501886|
|   Brazil|2501094|
|    Japan|2498267|
|Australia|2498353|
+---------+-------+

⏱️  Time after coalesce: 11.46 seconds


In [9]:
from pyspark.storagelevel import StorageLevel

# Different cache levels
print("💾 Cache Level Options:")
print("   • MEMORY_ONLY - Store in memory only")
print("   • MEMORY_AND_DISK - Spill to disk if memory full")
print("   • DISK_ONLY - Store on disk only")
print("   • MEMORY_ONLY_SER - Memory with serialization (less space, more CPU)")

# Time a query without cache
start = time.time()
df_sales.groupBy("country").agg(sum("final_amount")).collect()
print(f"\n⏱️  First run (no cache): {time.time()-start:.2f} seconds")

# Cache explicitly
df_sales.unpersist()  # Clear previous cache
df_sales.persist(StorageLevel.MEMORY_AND_DISK)

# Force cache
df_sales.count()

# Time same query with cache
start = time.time()
df_sales.groupBy("country").agg(sum("final_amount")).collect()
print(f"⏱️  Second run (with cache): {time.time()-start:.2f} seconds")
print("✅ Notice the speedup!")



💾 Cache Level Options:
   • MEMORY_ONLY - Store in memory only
   • MEMORY_AND_DISK - Spill to disk if memory full
   • DISK_ONLY - Store on disk only
   • MEMORY_ONLY_SER - Memory with serialization (less space, more CPU)


26/03/10 22:40:17 WARN MemoryStore: Not enough space to cache rdd_33_2 in memory! (computed 10.6 MiB so far)
26/03/10 22:40:17 WARN MemoryStore: Not enough space to cache rdd_33_3 in memory! (computed 5.4 MiB so far)
26/03/10 22:40:20 WARN MemoryStore: Not enough space to cache rdd_33_1 in memory! (computed 56.7 MiB so far)
26/03/10 22:40:21 WARN MemoryStore: Not enough space to cache rdd_33_6 in memory! (computed 20.8 MiB so far)
26/03/10 22:40:21 WARN MemoryStore: Not enough space to cache rdd_33_7 in memory! (computed 20.8 MiB so far)
                                                                                


⏱️  First run (no cache): 12.36 seconds


26/03/10 22:44:52 WARN BlockManager: Block rdd_242_9 could not be removed as it was not found on disk or in memory
26/03/10 22:44:52 WARN BlockManager: Block rdd_242_8 could not be removed as it was not found on disk or in memory
26/03/10 22:44:52 ERROR Executor: Exception in task 9.0 in stage 77.0 (TID 427)
java.lang.OutOfMemoryError: Java heap space
	at java.base/java.lang.Integer.valueOf(Integer.java:1059)
	at scala.runtime.BoxesRunTime.boxToInteger(BoxesRunTime.java:67)
	at scala.runtime.java8.JFunction0$mcI$sp.apply(JFunction0$mcI$sp.java:23)
	at org.apache.spark.sql.catalyst.util.MathUtils$.withOverflow(MathUtils.scala:83)
	at org.apache.spark.sql.catalyst.util.MathUtils$.toIntExact(MathUtils.scala:68)
	at org.apache.spark.sql.catalyst.util.SparkDateTimeUtils.localDateToDays(SparkDateTimeUtils.scala:152)
	at org.apache.spark.sql.catalyst.util.SparkDateTimeUtils.localDateToDays$(SparkDateTimeUtils.scala:152)
	at org.apache.spark.sql.catalyst.util.SparkDateTimeUtils$.localDateToDay

Py4JJavaError: An error occurred while calling o42.count.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 9 in stage 77.0 failed 1 times, most recent failure: Lost task 9.0 in stage 77.0 (TID 427) (10.0.2.15 executor driver): java.lang.OutOfMemoryError: Java heap space
	at java.base/java.lang.Integer.valueOf(Integer.java:1059)
	at scala.runtime.BoxesRunTime.boxToInteger(BoxesRunTime.java:67)
	at scala.runtime.java8.JFunction0$mcI$sp.apply(JFunction0$mcI$sp.java:23)
	at org.apache.spark.sql.catalyst.util.MathUtils$.withOverflow(MathUtils.scala:83)
	at org.apache.spark.sql.catalyst.util.MathUtils$.toIntExact(MathUtils.scala:68)
	at org.apache.spark.sql.catalyst.util.SparkDateTimeUtils.localDateToDays(SparkDateTimeUtils.scala:152)
	at org.apache.spark.sql.catalyst.util.SparkDateTimeUtils.localDateToDays$(SparkDateTimeUtils.scala:152)
	at org.apache.spark.sql.catalyst.util.SparkDateTimeUtils$.localDateToDays(SparkDateTimeUtils.scala:607)
	at org.apache.spark.sql.c